# V7_A_N11 — Agricultural Census and Frame Intelligence

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft v0.1. This notebook supports learning and review; it does not authorize an operational decision.

## Purpose and authority boundary
Use transparent diagnostics to prioritize **frame verification**. Official frame adoption remains with the competent statistical authority.

In [1]:
import numpy as np, pandas as pd
rng=np.random.default_rng(71)
n=180
df=pd.DataFrame({'ea_id':[f'EA{i:03d}' for i in range(n)],'region':rng.choice(['North','Central','South'],n,p=[.3,.4,.3]),'listed_holdings':rng.poisson(42,n),'admin_holdings':rng.poisson(45,n),'months_since_update':rng.integers(1,37,n),'gps_match_rate':rng.uniform(.72,1,n)})
df.head()

## Data contract
Entity: enumeration area (EA). Required keys: EA identifier and region. Diagnostic variables have different meanings and update cycles; none is ground truth by itself.

In [2]:
required=['ea_id','region','listed_holdings','admin_holdings','months_since_update','gps_match_rate']
assert df['ea_id'].is_unique and set(required)<=set(df)
assert df['gps_match_rate'].between(0,1).all()
print('ROWS',len(df),'DUPLICATE_KEYS',df.ea_id.duplicated().sum())

ROWS 180 DUPLICATE_KEYS 0


## Transparent discrepancy indicators
We compare listing and administrative counts, age of the frame, and geographic matching. These are verification signals—not automatic corrections.

In [3]:
df['count_gap_rate']=(df.admin_holdings-df.listed_holdings).abs()/df[['admin_holdings','listed_holdings']].max(axis=1).clip(lower=1)
df['staleness']=df.months_since_update/36
df['gps_gap']=1-df.gps_match_rate
df[['count_gap_rate','staleness','gps_gap']].describe().round(3)

## Baseline risk score
A published weighted rule is preferable to an opaque model until a stronger method demonstrates measurable benefit. Weights are governance assumptions and must be sensitivity-tested.

In [4]:
weights={'count_gap_rate':.45,'staleness':.30,'gps_gap':.25}
df['risk_score']=sum(df[k]*v for k,v in weights.items())
df['risk_band']=pd.cut(df.risk_score,[-np.inf,.25,.45,np.inf],labels=['routine','review','priority'])
df.risk_band.value_counts()

## Equity and operational capacity
A national ranking can concentrate fieldwork in large or data-rich regions. We therefore select within region and cap workload.

In [5]:
quota=5
selected=(df.sort_values(['region','risk_score'],ascending=[True,False]).groupby('region',group_keys=False).head(quota))
assert selected.groupby('region').size().eq(quota).all()
selected[['ea_id','region','risk_score','risk_band']].sort_values('risk_score',ascending=False).head(10).round(3)

## Sensitivity check
If a small change in weights changes many selected EAs, the prioritization is unstable and should be reviewed.

In [6]:
alt=.30*df.count_gap_rate+.45*df.staleness+.25*df.gps_gap
alt_ids=set(df.assign(alt=alt).sort_values(['region','alt'],ascending=[True,False]).groupby('region').head(quota).ea_id)
base_ids=set(selected.ea_id)
overlap=len(base_ids&alt_ids)/len(base_ids)
print('SELECTION_OVERLAP',round(overlap,3))

SELECTION_OVERLAP 0.8


## Decision product
The output is a verification worklist with reason codes and a review date—not a revised official frame.

In [7]:
def reason(r):
 vals={'count discrepancy':r.count_gap_rate,'stale frame':r.staleness,'geographic mismatch':r.gps_gap}
 return max(vals,key=vals.get)
worklist=selected.assign(reason_code=selected.apply(reason,axis=1),status='PENDING HUMAN REVIEW')
assert (worklist.status=='PENDING HUMAN REVIEW').all()
worklist[['ea_id','region','risk_score','reason_code','status']].head()

## Exercises
1. Replace the regional quota with a workload budget. 2. Test a different staleness normalization. 3. Explain why administrative counts cannot automatically replace listing counts. 4. Draft a field-verification outcome code list.

## Solution guide
1. Assign visit costs and select under a total cost constraint while retaining geographic safeguards. 2. Compare capped linear, categorical, and nonlinear transforms; report selection stability. 3. Coverage, concepts, reference dates, and incentives differ; discrepancies require reconciliation. 4. At minimum: confirmed listing, new holding, inactive holding, boundary error, duplicate, unable to verify, and referral.

In [8]:
assert worklist.ea_id.is_unique
assert len(worklist)==15
assert worklist.risk_score.between(0,1).all()
print('V7_A_N11_REWORK_COMPLETE_EXECUTION_PASS')

V7_A_N11_REWORK_COMPLETE_EXECUTION_PASS
